In [ ]:
# import data handling tools
import os
import numpy as np
import pandas as pd
import seaborn as sns
sns.set_style('darkgrid')
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

from PIL import Image

# import Deep learning Libraries
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam, Adamax
from tensorflow.keras.metrics import categorical_crossentropy
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Activation, Dropout, BatchNormalization
from tensorflow.keras import regularizers
import cv2
# Ignore Warnings
import warnings
warnings.filterwarnings("ignore")

print ('modules loaded')

In [ ]:
dataset_path = "dataset"

train_path = os.path.join(dataset_path, "train")
val_path = os.path.join(dataset_path, "val")
test_path = os.path.join(dataset_path, "test")

In [ ]:
classes = os.listdir(train_path)
print("Classes:", classes)

In [ ]:
def count_images(folder):
    counts = {}
    for cls in os.listdir(folder):
        cls_path = os.path.join(folder, cls)
        counts[cls] = len(os.listdir(cls_path))
    return counts

train_counts = count_images(train_path)
val_counts = count_images(val_path)
test_counts = count_images(test_path)

print("Train:", train_counts)
print("Validation:", val_counts)
print("Test:", test_counts)

In [ ]:
plt.figure(figsize=(8,5))
sns.barplot(x=list(train_counts.keys()), y=list(train_counts.values()))
plt.title("Training Data Distribution")
plt.xlabel("Class")
plt.ylabel("Number of Images")
plt.xticks(rotation=30)
plt.show()

In [ ]:
records = []

for split in ['train','val','test']:
    split_path = os.path.join(dataset_path, split)

    for label in os.listdir(split_path):
        class_path = os.path.join(split_path, label)

        for img in os.listdir(class_path):
            records.append({
                "split": split,
                "label": label,
                "filepath": os.path.join(class_path, img)
            })

eda_df = pd.DataFrame(records)

print("Total images:", len(eda_df))
eda_df.head()

In [ ]:
sizes = []

sample_paths = eda_df['filepath'].sample(200)

for path in sample_paths:
    img = Image.open(path)
    sizes.append(img.size)

size_df = pd.DataFrame(sizes, columns=['width','height'])

print(size_df.describe())

In [ ]:
sample_img = np.array(Image.open(eda_df['filepath'].iloc[0]))

print("Min pixel value:", sample_img.min())
print("Max pixel value:", sample_img.max())

In [ ]:
samples_per_class = 3

for label in classes:
    class_path = os.path.join(train_path, label)

    images = os.listdir(class_path)[:samples_per_class]

    plt.figure(figsize=(10,3))

    for i, img_name in enumerate(images):
        img = cv2.imread(os.path.join(class_path, img_name))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        plt.subplot(1,3,i+1)
        plt.imshow(img)
        plt.title(label)
        plt.axis("off")

    plt.show()

In [ ]:
broken = []

for path in eda_df['filepath']:
    try:
        Image.open(path)
    except:
        broken.append(path)

print("Broken images:", len(broken))